# 0930 21일차

## 1. 채팅 모델

모델 회사마다 다른 API를 같은 사용법으로 감싼 LangChain 클래스

회사를 바꿔도 `invoke`로 부르고 같은 형태의 응답을 받으므로, 모델을 만드는 줄만 바꾸면 됨

1. `ChatOpenAI` : OpenAI 모델
2. `ChatGoogleGenerativeAI` : Gemini 모델

- `base_url`을 바꾸면 `ChatOpenAI`로 OpenAI 형식을 따르는 다른 서버(monogpt)에도 연결됨
- `temperature` : 답변의 무작위성. 0이면 매번 거의 같은 답, 높을수록 다양함. OpenAI는 0~2
- `invoke` 한 번은 독립된 요청이라 이전 질문을 기억하지 않음. "내가 누구게?"에는 알 수 없다고 답함

### 1-1. AIMessage

`invoke`가 돌려주는 응답 객체

1. `content` : 모델이 보낸 원본. 문자열이거나 블록 리스트
2. `.text` : 텍스트만 뽑은 문자열
3. `response_metadata` : 모델 이름, 종료 이유(`finish_reason`), 토큰 사용량
4. `usage_metadata` : 입력·출력·전체 토큰 수

- 객체를 통째로 출력하면 메타데이터까지 나와서 답변만 볼 때는 `content`나 `.text`
- `content`가 블록 리스트로 오는 모델도 `.text`는 항상 문자열

## 2. PromptTemplate

`{변수}` 자리를 비워 둔 프롬프트 틀

질문마다 문장 전체를 새로 쓰지 않고, 바뀌는 부분만 dict로 넘김

1. 변수 한 개 : `{country}의 수도는 어디인가요?`
2. 변수 여러 개 : `{topic}`과 `{how}`처럼 자리를 여러 개 두고 dict에 키를 모두 넣음
3. 여러 줄 템플릿 : 역할(10년차 영어 선생님), 상황, 출력 형식을 한 틀에 넣음

- dict의 키 이름이 `{}` 안의 이름과 같아야 함
- 템플릿에 있는 변수가 dict에 없으면 `KeyError`
- 출력 형식을 `[FORMAT]`처럼 적어 두면 답변이 그 양식을 따름

## 3. LCEL

`|`로 단계를 이어 붙여 체인을 만드는 LangChain 문법 (LangChain Expression Language)

프롬프트 채우기 → 모델 호출 → 응답 다듬기를 `chain.invoke` 한 번으로 실행

```python
chain = prompt | model | output_parser
chain.invoke({'topic': '양자컴퓨터 학습 원리'})
```

| 단계 | 받는 것 | 내보내는 것 |
|---|---|---|
| `prompt` | dict | 변수가 채워진 프롬프트 |
| `model` | 프롬프트 | `AIMessage` |
| `output_parser` | `AIMessage` | 문자열 |

1. 앞 단계의 출력이 다음 단계의 입력이 됨
2. `invoke`에 넘기는 dict는 맨 앞 `prompt`가 받음

- pmo(prompt - model - output), plp(prompt - llm - parser)는 같은 순서를 부르는 말

## 4. 출력 파서

모델 응답(`AIMessage`)을 원하는 형태로 바꾸는 체인의 마지막 단계

체인이 모델에서 끝나면 `AIMessage`가 나와서 매번 `.content`를 꺼내야 함

1. `StrOutputParser` : 응답에서 텍스트만 문자열로 뽑음

- 파서가 없는 체인은 `response.content`, 있는 체인은 `response`를 그대로 출력
- parser : 분석하다. 응답을 다듬어 원하는 형태로 만든다는 뜻